# Interactive LLM Evaluation & Raw Sample Testing

This notebook provides a lightweight interactive environment to:
1. **Select a Model**: Choose from configured models (`models.json`), local folders, or Hugging Face repo IDs.
2. **Select a Benchmark**: Choose evaluation datasets (`gsm8k`, `mmlu`, `hellaswag`, `truthfulqa`).
3. **Inspect Raw Samples**: View unformatted dataset items alongside formatted evaluation prompts.
4. **Test & View Raw Responses**: Run model inference and examine raw generation outputs, thought reasoning tags, extracted answers, and evaluation scoring.
5. **Test Custom Prompts**: Send free-form prompts directly to the model.
6. **Clean Up Memory**: Easily clear GPU CUDA VRAM.

In [ ]:
import sys
import json
from pathlib import Path

# Add project root to sys.path
WORKSPACE_ROOT = Path(".").resolve()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from eval_core import (
    load_models_config,
    resolve_model_path,
    load_hf_model_and_tokenizer,
    run_hf_batch_inference,
    discover_available_datasets,
    extract_think_part,
    extract_final_answer,
    clear_gpu_vram
)
from evaluators import get_evaluator

print("✓ Core evaluation modules loaded successfully.")

In [ ]:
# Display available models in models.json
models_cfg = load_models_config()
print("Configured Models in models.json:")
for key, info in models_cfg.items():
    aliases_str = ", ".join(info.get("aliases", []))
    print(f" - {key:<15} ({info.get('description', '')}) | Aliases: [{aliases_str}] | Repo: {info.get('hf_repo', '')}")

# ==============================================================================
# 1) SELECT MODEL KEY OR PATH
# ==============================================================================
SELECTED_MODEL = "qwen3.5-2b"  # Available options: "qwen2.5-0.5b", "qwen3.5-2b", "qwen2.5-3b", "gemma-2-2b", "gemma-4-e2b"

resolved_model_path = resolve_model_path(SELECTED_MODEL)
print(f"\n[SELECTED MODEL]: {SELECTED_MODEL}")
print(f"[RESOLVED PATH ]: {resolved_model_path}")

# Discover benchmark datasets in ./data/
available_benchmarks = discover_available_datasets()
print(f"\nAvailable Benchmarks in ./data/: {available_benchmarks}")

# ==============================================================================
# 2) SELECT BENCHMARK DATASET
# ==============================================================================
SELECTED_BENCHMARK = "truthfulqa"  # Available options: "gsm8k", "mmlu", "hellaswag", "truthfulqa"
print(f"[SELECTED BENCHMARK]: {SELECTED_BENCHMARK}")

In [ ]:
# Load selected dataset
data_path = WORKSPACE_ROOT / "data" / f"{SELECTED_BENCHMARK}.json"

if not data_path.exists():
    print(f"❌ Dataset file '{data_path}' not found. Run download_datasets.py first!")
else:
    with open(data_path, "r", encoding="utf-8") as f:
        dataset_items = json.load(f)

    evaluator = get_evaluator(SELECTED_BENCHMARK)
    print(f"✓ Loaded {len(dataset_items)} items from '{SELECTED_BENCHMARK}.json'")
    
    # Inspect sample item in raw JSON format vs formatted evaluation prompt
    INSPECT_INDEX = 0
    raw_item = dataset_items[INSPECT_INDEX]
    formatted_prompt = evaluator.format_prompt(raw_item)
    
    print("\n" + "="*75)
    print(f"RAW DATASET ITEM (Index {INSPECT_INDEX}):")
    print("="*75)
    print(json.dumps(raw_item, indent=2))
    
    print("\n" + "="*75)
    print("FORMATTED EVALUATION PROMPT (Sent to Model):")
    print("="*75)
    print(formatted_prompt)

In [ ]:
# Set testing range and parameters
START_SAMPLE_INDEX = 0   # Starting sample index in dataset
NUM_TEST_SAMPLES = 1     # Number of consecutive samples to evaluate
MAX_NEW_TOKENS = 512     # Maximum generation token count

if 'dataset_items' in locals() and dataset_items:
    test_batch = dataset_items[START_SAMPLE_INDEX : START_SAMPLE_INDEX + NUM_TEST_SAMPLES]
    evaluator = get_evaluator(SELECTED_BENCHMARK)
    prompts = [evaluator.format_prompt(item) for item in test_batch]
    
    print(f"Running inference for {len(prompts)} sample(s) on model '{SELECTED_MODEL}'...")
    inference_results = run_hf_batch_inference(resolved_model_path, prompts, max_new_tokens=MAX_NEW_TOKENS)
    
    for idx, (item, prompt, res) in enumerate(zip(test_batch, prompts, inference_results)):
        sample_idx = START_SAMPLE_INDEX + idx
        raw_response = res["text"]
        think_reasoning = extract_think_part(raw_response)
        extracted_ans = extract_final_answer(raw_response)
        
        is_pass, pred_val, score_reason = evaluator.score_item(item, extracted_ans)
        status_str = "✅ PASS" if is_pass else "❌ FAIL"
        
        print("\n" + "#"*75)
        print(f" SAMPLE {sample_idx + 1} / {len(dataset_items)} | RESULT: {status_str}")
        print(f" Latency: {res['sample_latency_ms']} ms | Speed: {res['aggregate_tps']} tok/s")
        print("#"*75)
        print(f"\n[1. RAW DATASET ITEM]:\n{json.dumps(item, indent=2)}")
        print(f"\n[2. PROMPT PASSED TO MODEL]:\n{prompt}")
        print(f"\n[3. RAW UNFILTERED MODEL RESPONSE]:\n{raw_response}")
        print(f"\n[4. THOUGHT / REASONING BLOCK]:\n{think_reasoning}")
        print(f"\n[5. EXTRACTED ANSWER]:\n{extracted_ans}")
        print(f"\n[6. EVALUATION SCORE REASON]:\n{score_reason}")
        print("="*75)

In [ ]:
# Test any custom raw prompt directly on the selected model
CUSTOM_RAW_PROMPT = """Question: Janet has 3 apples. She buys 2 more bags with 5 apples each. How many apples does she have in total?
Show your work step-by-step."""

print(f"Testing custom prompt on '{SELECTED_MODEL}'...")
custom_res = run_hf_batch_inference(resolved_model_path, [CUSTOM_RAW_PROMPT], max_new_tokens=512)[0]

raw_output = custom_res["text"]
think_output = extract_think_part(raw_output)
clean_output = extract_final_answer(raw_output)

print("\n" + "="*75)
print("CUSTOM PROMPT:")
print("="*75)
print(CUSTOM_RAW_PROMPT)
print("\n" + "="*75)
print("RAW UNFILTERED MODEL RESPONSE:")
print("="*75)
print(raw_output)
print("\n" + "="*75)
print("REASONING (<think>):")
print("="*75)
print(think_output)
print("\n" + "="*75)
print("EXTRACTED ANSWER:")
print("="*75)
print(clean_output)
print(f"\n[Performance]: Latency {custom_res['sample_latency_ms']} ms | Speed {custom_res['aggregate_tps']} tok/s")

In [ ]:
# Release PyTorch CUDA memory and garbage collect
clear_gpu_vram()
print("✓ GPU VRAM cache cleared.")